In [1]:
from datasets import load_dataset
import numpy as np
from collections import Counter
import os

# temporary use of
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

In [2]:
ds = load_dataset("afmck/text8") # https://huggingface.co/datasets/afmck/text8

## The Skip-gram Model
the objective of the Skip-gram model is to maximize the average log probability

$$
\frac{1}{T} \sum_{t=1}^{T} \sum_{\substack{-c \le j \le c,j \ne 0}} \log p(w_{t+j} \mid w_t)
$$
- $c$ is the size of the training context (which can be a function of the center word $w_{t}$)

### Softmax

The basic Skip-gram formulation defines $p(w_{t+j} \mid w_t)$ using the softmax function:

$$
p(w_O \mid w_I) = \frac{\exp\left({v'_{w_O}}^{\top} v_{w_I}\right)}{\sum_{w=1}^{W} \exp\left({v'_w}^{\top} v_{w_I}\right)}
$$
- $v_{w}$ and $v'_{w}$ are the “input” and “output” vector representations of $w$, and $W$ is the number of words in the vocabulary.<br>
- This formulation is impractical because the cost of computing $\nabla \log p(w_O \mid w_I)$ is proportional to $W$, which is often very large ($10^5$–$10^7$ terms).

### Hierarchical Softmax

Then the hierarchical softmax defines $p(w_O \mid w_I)$ as follows:

$$
p(w \mid w_I) = \prod_{j=1}^{L(w)-1} \sigma \left( [\!\left[ n(w, j+1) = ch(n(w, j)) \right]\!\right] \cdot {v'}_{n(w,j)}^{\top} v_{w_I} )
$$

- Let $n(w, j)$ be the $j$-th node on the path from the root to $w$
- let $L(w)$ be the length of this path, so $n(w, 1) = root$ and $n(w, L(w)) = w$
- $\sigma(x) = {1} / {(1 + \exp(-x))}$. It can be verified that $\sum_{w=1}^{W} p(w \mid w_I) = 1$
- let $ch(n)$ be an arbitrary fixed child of $n$
- let $ \left[\!\left[ x \right]\!\right] $ be 1 if $x$ is true and -1 otherwise

### Negative Sampling

An alternative to the hierarchical softmax is Noise Contrastive Estimation (NCE)

We define Negative sampling (NEG) by the objective 

$$
\log \sigma\left({v'}_{w_O}^{\top} v_{w_I}\right) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)} \left[ \log \sigma\left(-{v'}_{w_i}^{\top} v_{w_I}\right) \right]
$$

which is used to replace every $\log P (w_O | w_I )$ term in the Skip-gram objective

- values of k in the range 5–20 are useful for small training datasets
- for large datasets the k can be as small as 2–5

> The main difference between the Negative sampling and NCE is that NCE needs both samples and the numerical probabilities of the noise distribution, while Negative sampling uses only samples

#### Noise Distribution

Both NCE and NEG have the noise distribution $P_n(w)$ as a free parameter

- unigram distribution $U(w)$ raised to the 3/4rd power (i.e., $U(w)^{3/4}/Z$) outperformed significantly the unigram and the uniform distributions

### Subsampling of Frequent Words

To counter the imbalance between the rare and frequent words, we used a simple subsampling approach: each word $w_i$ in the training set is discarded with probability computed by the formula

$$
P(w_i) = 1 - \sqrt{ \frac{t}{f(w_i)} }
$$

- $f(w_i)$ is the frequency of word $w_i$
- $t$ is a chosen threshold, typically around $10^{-5}$

We chose this subsampling formula because it aggressively subsamples words whose frequency is greater than $t$ while preserving the ranking of the frequencies

In [3]:
def split_data(ds):
    train, test, validation = ds["train"], ds["test"], ds["validation"]
    split_text = lambda dataset: [x for x in dataset["text"][0].split() if x]
    train = split_text(train)
    test = split_text(test)
    validation = split_text(validation)
    return train, test, validation

train, test, validation = split_data(ds)

In [4]:
def subsampling(words, threshold=1e-5):
    counts = Counter(words)
    total_count = len(words)
    # P(w) = (sqrt(f(w)/t) + 1) * (t/f(w))
    new_words = []
    for w in words:
        freq = counts[w] / total_count
        p_keep = (np.sqrt(freq / threshold) + 1) * (threshold / freq)
        if np.random.random() < p_keep:
            new_words.append(w)
    return new_words

In [5]:
def create_unigram_table(word_counts, vocab, table_size=100_000_000):
    freqs = np.array([word_counts[w] for w in vocab])
    pow_freqs = freqs**0.75
    probs = pow_freqs / sum(pow_freqs)
    
    # Tabloyu indekslerle doldur
    table = np.zeros(table_size, dtype=np.int32)
    count = 0
    for idx, p in enumerate(probs):
        n = int(p * table_size)
        table[count : count + n] = idx
        count += n
    return table
# neg_samples = table[np.random.randint(0, table_size, size=k)]

In [6]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # Eğer GPU (CUDA) kullanıyorsan
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Multi-GPU için

    # İşlemleri deterministik (kesin) hale getir
    # Not: Bu işlemler eğitimi %5-10 yavaşlatabilir ama sonuçlar hep aynı çıkar
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Eğitime başlamadan önce çağır
set_seed(42) # Klasik 42, istersen başka sayı yap

In [7]:
# Andrej Karpathy: https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part4_backprop.ipynb
def cmp(s, dt, t):
    dt = torch.as_tensor(dt).float()
    t = torch.as_tensor(t).detach().float()
    
    ex = torch.all(dt == t).item()
    app = torch.allclose(dt, t, atol=1e-7)
    maxdiff = (dt - t).abs().max().item()
    
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [8]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [9]:
# Frequency of the words
full_counts = Counter(train)
# Discard words occured less than 5 times & subsamble
train = [w for w in train if full_counts[w] >= 5]
train = subsampling(train)

# Create alphabetic vocabulary
vocab = sorted(list(set(train)))
# word to index & index to word operations
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab) # Vocabulary size

# Frequency of the words after discarding & subsampling
word_counts = {w: full_counts[w] for w in vocab}

# Unigram table for Negative Sampling
unigram_table = create_unigram_table(word_counts, vocab)

In [10]:
print(f'Sözlük Boyutu:        {vocab_size:,}')
print(f'Toplam kelime sayısı: {len(train):,}')

Sözlük Boyutu:        67,427
Toplam kelime sayısı: 4,999,119


In [11]:
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.u_embeddings = nn.Embedding(vocab_size, emb_dim) # Center words
        self.v_embeddings = nn.Embedding(vocab_size, emb_dim) # Context & Negative words
        
        # Ağırlıkları başlatma
        self.u_embeddings.weight.data.uniform_(-1.0, 1.0)
        self.v_embeddings.weight.data.uniform_(-1.0, 1.0)

    def forward(self, pos_u, pos_v, neg_v):
        # pos_u: [batch_size]
        # pos_v: [batch_size]
        # neg_v: [batch_size, k]

        self.emb_u = self.u_embeddings(pos_u)
        self.emb_v = self.v_embeddings(pos_v)
        self.emb_neg = self.v_embeddings(neg_v)
        
        B, K, D = self.emb_neg.shape
        
        # Pozitif Skor: dot(u, v)
        self.pos_score = torch.sum(torch.mul(self.emb_u, self.emb_v))
        self.pos_loss = F.logsigmoid(self.pos_score)
        
        # Negatif Skor: u * neg_v
        # [B, K, D] @ [B, D, 1] -> [B, K, 1]
        self.neg_score = torch.bmm(self.emb_neg, self.emb_u.view(B, D, 1)).view(B, K)
        self.neg_loss = torch.sum(F.logsigmoid(-self.neg_score))
        
        return -(self.pos_loss + self.neg_loss)

In [12]:
def first_idx(word):
    for i, data in enumerate(train):
        if data == word:
            return i
    raise KeyError(f"Cannot find {word} in dataset")

In [13]:
def get_negative_samples(size, blacklist):
    samples = []
    while len(samples) < size:
        # unigram_table'dan rastgele bir index seç
        candidate = np.random.choice(unigram_table)
        
        # Eğer seçilen kelime blacklist'te (pos_u veya pos_v) yoksa listeye ekle
        if candidate not in blacklist:
            samples.append(int(candidate))
            
    return samples

In [42]:
# --- Hiperparametreler ---
EMB_DIM = 512
K_NEG = 5
window = 2

# numpy default olarak float64 kullandığı için
# torch'u da float64'e default ediyoruz
torch.set_default_dtype(torch.float64)
# Referans olması amacıyla torch modelimiz
model = SkipGramModel(vocab_size, EMB_DIM)

word = "king" # hesap yapacağımız kelime
i = first_idx(word) # train datasetinde kaçıncı index'te karşılaşıyoruz
target_idx = word2idx[word] # word2idx tablosunda hangi index'e sahip
context_positions = [j for j in range(i-window, i+window+1) if j != i]
pairs = [(target_idx, word2idx[train[context_idx]]) for context_idx in context_positions]

print(f'king kelimesi için aralık: "{' '.join(train[i-window: i+window+1])}"')
print(f"bağlam ikilileri: {", ".join([f"{idx2word[u]}-{idx2word[v]}" for u, v in pairs])}")

pos_u, pos_v = pairs[0] # şimdilik sadece ilk hedef için modeli deneyelim

# 'king' kelimesi ile alakalı olmayan (rastgele) 5 kelime seçelim.
# Şimdi bu fonksiyonu kullanarak yeni neg_v listeni oluşturalım:
neg_v = get_negative_samples(size=K_NEG, blacklist=[pos_u, pos_v])

print(f"Hedef  (u): {idx2word[pos_u]}")
print(f"Bağlam (v): {idx2word[pos_v]}")
print(f"Negatif Örnekler: {", ".join([f"{idx2word[idx]}({idx})" for idx in neg_v])}")

king kelimesi için aralık: "archons ruler king anarchism political"
bağlam ikilileri: king-archons, king-ruler, king-anarchism, king-political
Hedef  (u): king
Bağlam (v): archons
Negatif Örnekler: quotation(49359), lombards(36147), salts(53112), carrying(9670), based(5580)


In [57]:
neg_v_torch = torch.tensor(neg_v).long().view((1,-1))
pos_u_torch = torch.tensor(pos_u).view((1, -1))
pos_v_torch = torch.tensor(pos_v).view((1, -1))

# Ağırlık matrislerini kopyala
W_u = model.u_embeddings.weight.detach().clone().numpy()
W_v = model.v_embeddings.weight.detach().clone().numpy()

# Sözlükteki her bir kelime, 512 boyutlu bir vektör (embedding) ile temsil edilir.
# Haliyle 'W_v' matrisi 67,427 x 512 boyutunda bir tablo olur.

# Forward
model_loss = model(pos_u_torch, pos_v_torch, neg_v_torch)

# modelin yaptığı işlemleri 'manuel' olarak yapıp kontrol edelim.

# öncelikle hedef, bağlam ve negatif örneklerimiz için
# embedding vektörlerini alalım. Burada hedef kelime için
# ayrı vektör, bağlam ve negatif kelime için ayrı vektör kullanacağız.
emb_u = W_u[pos_u]
emb_v = W_v[pos_v]
emb_neg = W_v[neg_v]

# Pozitif skoru hesaplayalım, bu skor seçtiğimiz kelimenin yakınlarındaki
# kelimeler ile arasındaki 'yakınlığı' göstermektedir. Yani hedef ve bağlam
# kelimeleri birbirine ne kadar yakınsa bu skor o kadar yüksek olacak.
# Bu skora 
# model: torch.sum(torch.mul(self.emb_u, self.emb_v))
pos_score = 0
for i in range(EMB_DIM):
    pos_score += emb_u[i] * emb_v[i]

# Pozitif skorumuzu olasılığa çevirelim
# pos_score ne kadar yüksekse, probability o kadar 1'e yaklaşır.
probability = sigmoid(pos_score)

# Log-Sigmoid değerini hesaplayalım (pos_loss)
# Modelin amacı bu olasılığı 1 yapmaktır. 
# Eğer olasılık 1 ise log(1) = 0 olur (yani kayıp sıfırlanır).
pos_loss = np.log(probability)

# Şimdi aynı ihtimal/kayıp hesabını negatif kelimeler için de hesaplayalım.
# Bağlam olarak tek bir kelime kullandığımız için bir adet sayı elde etmiştik.
# Negatif olarak 5 sayı seçtiğimiz için bu 5 sayı için de ihtimal hesabını
# yapıyoruz, ancak burada aslında bu kelimenin gelmesini istemediğimiz için 
# elde skor değerini -1 ile çarpıyoruz. Yani skor ne kadar yüksek ise o kadar kötü

# model: torch.bmm(self.emb_neg, self.emb_u.view(B, D, 1)).view(B, K)
neg_score = np.zeros((K_NEG)) # her negatif kelime için bir adet skor değeri olacak
for n in range(K_NEG):
    for i in range(EMB_DIM):
        neg_score[n] += emb_u[i] * emb_neg[n][i]

# model: torch.sum(F.logsigmoid(-self.neg_score), dim=1)
neg_loss = 0
for n in range(K_NEG):
    neg_loss += np.log(sigmoid(-1 * neg_score[n]))

# Son olarak da hesapladığımız bu 2 kayıp değerini alıp nihai kayıp
# olarak kullanıyoruz: -(self.pos_loss + self.neg_loss)
loss = -(neg_loss+pos_loss)

###########################################
# Backprop 
###########################################

# Sıra backprop'u hesaplamada. Amacımız merkez, bağlam ve negatif
# kelimeler için hesapladığımız loss'a göre türevini bulmak.
grad_u = np.zeros(EMB_DIM)
grad_v = np.zeros(EMB_DIM)
grad_neg = np.zeros((K_NEG, EMB_DIM))

# bağlam için türev almak oldukta kolay olacak, çünkü loss dekleminde sadece
# pozitif skoru aldığımız yerde geçiyor. Hatta bağlam vektörünün loss hesabındaki
# etkisini kabaca şu şekilde yazabiliriz: log(sigmoid(emb_u * emb_v))

# öncelikle yapılan son işlem olan -(pos_loss+neg_loss) işlemindeki kısmi türevi alalım.
dloss_dlog = -1
# logaritmanın türevi: d(log(x))/dx = 1/x
# burada x = probability (yani sigmoid sonucu)
dlog_dprob = 1 / probability
# sigmoid'in türevi: d(sigmoid(x))/dx = sigmoid(x) * (1 - sigmoid(x))
dprob_dscore = probability * (1 - probability)
# zincir kuralı: -1 * (1/prob) * (prob * (1-prob))
dloss_dscore = probability - 1
# son olarak da merkez vektörü ile çarptığımız için 
for i in range(EMB_DIM):
    grad_v[i] = dloss_dscore * emb_u[i]

# negatif örnekler için de türev almak çok zor değil, buradaki fark birden fazla
# vektör için aynı işlemleri yapacağız. neg_loss kısmını hatırlarsak:
# her bir negatif örnek için log(sigmoid(-1 * neg_score)) değerlerini toplamıştık.

neg_score_grad = np.zeros(K_NEG)
for n in range(K_NEG):
    # negatif skorumuzun olasılığını alalım (sigmoid(x))
    # modelde skorları -1 ile çarpıp topladığımızı unutmayalım.
    neg_score_grad[n] = sigmoid(neg_score[n])
    
    # zincir kuralını negatif skor için uygulayalım:
    # dloss/dneg_loss = -1 (en dıştaki eksi)
    # dlog/dprob = 1 / sigmoid(-neg_score)
    # dprob/dscore = sigmoid(-neg_score) * (1 - sigmoid(-neg_score)) * (-1)
    # bu çarpmalar sonucunda (negatif işaretler birbirini götürür) sadeleşen türev:
    dloss_dneg_score = neg_score_grad[n]
    
    # negatif bağlam vektörü (v_neg) için gradyan:
    # dloss/dscore * merkez_vektörü
    for i in range(EMB_DIM):
        grad_neg[n][i] = dloss_dneg_score * emb_u[i]
        
    # merkez vektörü (u) her bir negatif örnekten de etkilendiği için
    # buradaki gradyanları u üzerinde biriktirmemiz (accumulate) gerekiyor:
    for i in range(EMB_DIM):
        grad_u[i] += dloss_dneg_score * emb_neg[n][i]

# son olarak merkez vektörüne (u), demin hesapladığımız pozitif 
# bağlamdan gelen etkiyi de ekleyelim ki grad_u tamamlansın:
for i in range(EMB_DIM):
    grad_u[i] += dloss_dscore * emb_v[i]

# modele gradyanları saklamasını söylüyoruz.
model.emb_u.retain_grad()
model.emb_v.retain_grad()
model.emb_neg.retain_grad()
model.pos_score.retain_grad()
model.neg_score.retain_grad()
model.pos_loss.retain_grad()
model.neg_loss.retain_grad()
model_loss.backward()

cmp('pos_score', model.pos_score, pos_score)
cmp('pos_loss', model.pos_loss, pos_loss)
cmp('neg_score', model.neg_score, neg_score)
cmp('neg_loss', model.neg_loss, neg_loss)
cmp('loss', model_loss, loss)
cmp('emb_u_grad', model.emb_u.grad, grad_u)
cmp('emb_v_grad', model.emb_v.grad, grad_v)
cmp('emb_neg_grad', model.emb_neg.grad, grad_neg)
cmp('pos_score_grad', model.pos_score.grad, dloss_dscore)
cmp('pos_loss_grad', model.pos_loss.grad, dloss_dlog)
cmp('neg_score_grad', model.neg_score.grad, neg_score_grad)
cmp('neg_loss_grad', model.neg_loss.grad, dloss_dlog)

pos_score       | exact: True  | approximate: True  | maxdiff: 0.0
pos_loss        | exact: True  | approximate: True  | maxdiff: 0.0
neg_score       | exact: True  | approximate: True  | maxdiff: 0.0
neg_loss        | exact: True  | approximate: True  | maxdiff: 0.0
loss            | exact: True  | approximate: True  | maxdiff: 0.0
emb_u_grad      | exact: True  | approximate: True  | maxdiff: 0.0
emb_v_grad      | exact: True  | approximate: True  | maxdiff: 0.0
emb_neg_grad    | exact: True  | approximate: True  | maxdiff: 0.0
pos_score_grad  | exact: True  | approximate: True  | maxdiff: 0.0
pos_loss_grad   | exact: True  | approximate: True  | maxdiff: 0.0
neg_score_grad  | exact: True  | approximate: True  | maxdiff: 0.0
neg_loss_grad   | exact: True  | approximate: True  | maxdiff: 0.0
